In [1]:
pip install transformers torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 71.8 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found existing installation: nvidia-curand-cu12 10.3.10.19
    Uninstalling nvidia-curand-cu12-

In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import classification_report
from tqdm import tqdm

train_json = "/kaggle/input/silver/silver_multi.json"
val_json = "/kaggle/input/sarcasm-data/multi_dev.json"
img_dir = "/kaggle/input/sarcasm-data/image/image/image"
test_json = "/kaggle/input/sarcasm-data/multi_test.json"

label2id = {"Non-sarcasm": 0, "Sarcasm": 1}
id2label = {0: "Non-sarcasm", 1: "Sarcasm"}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

class CLIPSarcasmDataset(Dataset):
    def __init__(self, json_file, img_dir):
        with open(json_file, "r", encoding="utf-8") as f:
            self.data = json.load(f)
        self.img_dir = img_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        caption = item["caption"]
        img_path = os.path.join(self.img_dir, item["image"])
        image = Image.open(img_path).convert("RGB")
        label = label2id[item["label"]]
        return {"image": image, "text": caption, "label": label}

def collate_clip(batch):
    images = [item["image"] for item in batch]
    texts = [item["text"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch])

    inputs = processor(text=texts, images=images, return_tensors="pt", padding=True, truncation=True, max_length=77)
    return {**inputs, "labels": labels}

train_dataset = CLIPSarcasmDataset(train_json, img_dir)
val_dataset = CLIPSarcasmDataset(val_json, img_dir)
test_dataset = CLIPSarcasmDataset(test_json, img_dir)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_clip)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_clip)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_clip)


class CLIPClassifier(nn.Module):
    def __init__(self, hidden_size=512, num_labels=2):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 256),
            nn.ReLU(),
            nn.Linear(256, num_labels)
        )

    def forward(self, img_emb, txt_emb):
        combined = torch.cat([img_emb, txt_emb], dim=1)
        return self.classifier(combined)

classifier = CLIPClassifier().to(device)
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

# Training
epochs = 4
for epoch in range(epochs):
    classifier.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} - Training"):
        with torch.no_grad():
            image_features = clip_model.get_image_features(batch["pixel_values"].to(device))
            text_features = clip_model.get_text_features(batch["input_ids"].to(device))

        logits = classifier(image_features, text_features)
        loss = loss_fn(logits, batch["labels"].to(device))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Train Loss: {total_loss / len(train_loader):.4f}")

    classifier.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Test Evaluation"):
            image_features = clip_model.get_image_features(batch["pixel_values"].to(device))
            text_features = clip_model.get_text_features(batch["input_ids"].to(device))
    
            logits = classifier(image_features, text_features)
            pred_labels = torch.argmax(logits, dim=1).cpu().numpy()
            true_labels = batch["labels"].cpu().numpy()
    
            preds.extend(pred_labels)
            targets.extend(true_labels)
    
    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average="macro")
    precision = precision_score(targets, preds, average="macro")
    recall = recall_score(targets, preds, average="macro")
    
    print(f"\n[TEST] Accuracy: {acc:.4f} | F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")

save_path = "saved_models/clip_classifier.pt"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(classifier.state_dict(), save_path)
print(f"[INFO] Classifier saved to {save_path}")

Epoch 1 - Training: 100%|██████████| 125/125 [00:35<00:00,  3.54it/s]


Train Loss: 0.6695


Test Evaluation: 100%|██████████| 13/13 [00:03<00:00,  3.41it/s]



[TEST] Accuracy: 0.4100 | F1: 0.4071 | Precision: 0.4443 | Recall: 0.4393


Epoch 2 - Training: 100%|██████████| 125/125 [00:35<00:00,  3.54it/s]


...


[TEST] Accuracy: 0.4550 | F1: 0.4397 | Precision: 0.4585 | Recall: 0.4527


Epoch 4 - Training: 100%|██████████| 125/125 [00:34<00:00,  3.57it/s]


Train Loss: 0.5646


Test Evaluation: 100%|██████████| 13/13 [00:03<00:00,  3.49it/s]


[TEST] Accuracy: 0.5050 | F1: 0.4786 | Precision: 0.4874 | Recall: 0.4858
[INFO] Classifier saved to saved_models/clip_classifier.pt


In [ ]:
print("\n--- Testing on Test Set ---")
classifier.eval()
clip_model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        labels = batch["labels"]
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)

        image_features = clip_model.get_image_features(pixel_values)
        text_features = clip_model.get_text_features(input_ids)
        
        logits = classifier(image_features, text_features)
        preds = torch.argmax(logits, dim=1).cpu()

        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print(classification_report(
    all_labels, all_preds,
    target_names=["Non-sarcasm", "Sarcasm"],
    digits=4
))



--- Testing on Test Set ---


Testing: 100%|██████████| 13/13 [00:03<00:00,  3.49it/s]

              precision    recall  f1-score   support

 Non-sarcasm     0.6636    0.5407    0.5959       135
     Sarcasm     0.3111    0.4308    0.3613        65

    accuracy                         0.5050       200
   macro avg     0.4874    0.4858    0.4786       200
weighted avg     0.5491    0.5050    0.5197       200

